## 2. Download Proxies Workflow

1. Packages
2. Comments
3. Settings
4. Area of Interest & Tiles
5. Compute Satellite Derived Bathymetry

### 1. Packages

In [1]:
# Generic packages
import folium
import geopandas as gpd
import numpy as np
import os
import sys
import time
import pickle
from tqdm import tqdm
from geojson import Feature, FeatureCollection, dump

# GEE specific packages
project = "cmems-sdb-11209821-002" #'bathymetry'
import ee
try:
    ee.Initialize(project=project)
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project=project)

# custom functionality import without requirement to pip install package
dir_path_ee_packages = os.path.join(os.path.expanduser('~'), 'Documents', 'GitHub', 'ee-packages-py') # path to local GitHub clone
sys.path.append(dir_path_ee_packages)
from eepackages.applications.bathymetry import Bathymetry
from eepackages import tiler

C:\Users\kras\AppData\Local\Temp\ipykernel_8296\3862242120.py:3: DeprecationWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas still uses PyGEOS by default. However, starting with version 0.14, the default will switch to Shapely. To force to use Shapely 2.0 now, you can either uninstall PyGEOS or set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In the next release, GeoPandas will switch to using Shapely by default, even if PyGEOS is installed. If you only have PyGEOS installed to get speed-ups, this switch should be smooth. However, if you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_pygeos.html).
  import geopandas as gpd


### 2. Comments

Acknowledgements & code references:
- https://github.com/openearth/eo-bathymetry/
- https://github.com/openearth/eo-bathymetry-functions/
- https://github.com/gee-community/ee-packages-py

In [2]:
# TODO list
# TODO: look if scale / crs does not influence the output used before exporting as we have differences between the GEE export and the local post-processed export

### 3. Settings

In [17]:
# Settings
run_mode = 'local'                      # Run mode, either 'local' or 'global'
project_name = 'roadmap_dynamicSDB_pilotsites'      # Name of the project AoI, or one in the folder
mode = 'intertidal_improved_10m_dynamics'  # Specify mode, either 'intertidal' or 'subtidal'
start_date = '2019-01-01'                  # Start date of the composites
stop_date = '2024-01-01'                   # End date of the composites
compo_int = 12                             # Composite interval [months]
compo_len = 12                             # Composite length [months]
scale = 10                                # Output resolution of the image [m]
crs = 'EPSG:4326'                          # Output projection of the image

# Tiling (see https://www.openearth.nl/rws-bathymetry/2019.html)
zoom_levels = [9, 10, 11] # list with zoom levels
zoom_level = zoom_levels[1] # zoom level to be used

# Directories and files
dir_path_base = r'p:\11209821-cmems-global-sdb'
dir_path_output = os.path.join(dir_path_base, '01_intertidal', '02_data', '05_calibrated', f'{mode}')                                                         # Output directory
#file_path_aoi = os.path.join(dir_path_base, '00_miscellaneous', 'AOI_upscale', '{}.geojson'.format(project_name.replace('_v2','')))                           # AOI file
file_path_aoi = os.path.join(dir_path_base, '00_miscellaneous', 'AOI_dynamics', '{}.geojson'.format(project_name))                                            # AOI file
file_path_mask = os.path.join(dir_path_base, '00_miscellaneous', 'Feasibility_maps', '2024', 'gebco_2024_latminus2_merge_result.parquet')                     # Mask file
file_path_mask_ed = os.path.join(dir_path_base, '00_miscellaneous', 'Feasibility_maps', '2024', 'gebco_2024_latminus2_merge_result_erosion_dilation.parquet') # Mask (erosion/dilation) file
file_path_tiles = os.path.join(dir_path_base, '00_miscellaneous', 'AOI_polygons_world', f'df_boxes_world_Z{zoom_level}_filtered_v2.parquet')                  # Tiles file
file_path_credentials = os.path.join(dir_path_base, '00_miscellaneous', 'KEYS', "cmems-sdb-11209821-002-d08744ac2a69.json") #'bathymetry-543b622ddce7.json'   # Cloud Storage credentials file
file_path_progress = os.path.join(dir_path_base, '01_intertidal', '02_data', '05_calibrated', 'progress_{}'.format(run_mode))                                 # progress dir

# Google Cloud Bucket
bucket = "cmems-isdb" #'cmems-sdb'

# Load Google credentials
if not file_path_credentials == '':  
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = file_path_credentials

# load GTSM & gebco data
#gtsm_col = ee.FeatureCollection('projects/bathymetry/assets/gtsm_waterlevels_2021_v2') # Loaded in bathymetry
#gebco_image = ee.Image('projects/bathymetry/assets/gebco_2023_hat_lat') # Loaded in bathymetry

### 4. Area of Interest & Tiles

In [18]:
# Read geometries
gdf_aoi = gpd.read_file(file_path_aoi)
gdf_mask = gpd.read_parquet(file_path_mask)
gdf_mask_ed = gpd.read_parquet(file_path_mask_ed)
gdf_tiles = gpd.read_parquet(file_path_tiles)

In [19]:
project_name = "dynamics_10m" 

if run_mode == 'local':   

    # Get mask where pixel value is 3.0
    gdf_mask = gdf_mask[gdf_mask['pixel_value'] == 3.0]
    gdf_mask_ed = gdf_mask_ed[gdf_mask_ed['pixel_value'] == 3.0]

    # Get mask that intersects with the area of interest
    gdf_mask = gdf_mask[gdf_mask.intersects(gdf_aoi.unary_union)]
    gdf_mask_ed = gdf_mask_ed[gdf_mask_ed.intersects(gdf_aoi.unary_union)]

    # Clip mask to area of interest
    gdf_mask = gpd.overlay(gdf_mask, gdf_aoi, how='intersection')
    gdf_mask_ed = gpd.overlay(gdf_mask_ed, gdf_aoi, how='intersection')

    # Get tiles that intersect with mask
    #gdf_tiles = gdf_tiles[gdf_tiles.intersects(gdf_mask_ed.unary_union)]
    gdf_tiles = gdf_tiles[gdf_tiles.intersects(gdf_aoi.unary_union)] # only for dynamics bathy to have all tiles 

    # Select tiles within bounds (France)
    #bounds = [-5.0, 45.5, 0, 47.5]
    #gdf_tiles = gdf_tiles.cx[bounds[0]:bounds[2], bounds[1]:bounds[3]]

    # Select specific tile
    #gdf_tiles = gdf_tiles[gdf_tiles['name'] == 'z10_x507_y363']
    #gdf_tiles = gdf_tiles[gdf_tiles['name'] == 'z10_x528_y331']

    # Sort tiles based on intertidal coverage
    gdf_tiles = gdf_tiles.sort_values(by='intertidal_coverage_ed', ascending=False)

    # Print tiles
    print('Number of tiles: {}'.format(len(gdf_tiles)))
    gdf_tiles.head(5)

    # export tiles
    # for idx, row in tqdm(gdf_tiles.iterrows(), total=len(gdf_tiles)):
    #     # Get tile geodataframe
    #     tile = gdf_tiles.loc[[idx]]
    #     print(type(tile["geometry"].values[0]), type(tile.name.values[0]), type(tile.id.values[0]), type(tile.tx.values[0]), type(tile.ty.values[0]), type(tile.zoom.values[0]))

    #     features = []
    #     features.append(Feature(geometry=tile.geometry.values[0], properties={"name": tile.name.values[0], "id":str(tile.id.values[0]), "tx":float(tile.tx.values[0]), "ty":float(tile.ty.values[0]), "zoom":str(int(float(tile.zoom.values[0])))}))
    #     feature_collection = FeatureCollection(features)
    #     feature_collection.crs = {"type": "name","properties": {"name": "epsg:4326"}} # default EE projection
    #     with open(os.path.join(dir_path_base, '00_miscellaneous', 'AOI_polygons_dynamics', tile.name.values[0] + ".geojson"), "w") as f: # geojson
    #         dump(feature_collection, f)

if run_mode == 'global' and project_name != "failed":

    # Get mask where pixel value is 3.0
    #gdf_mask = gdf_mask[gdf_mask['pixel_value'] == 3.0]
    #gdf_mask_ed = gdf_mask_ed[gdf_mask_ed['pixel_value'] == 3.0]

    # Get mask that intersects with the area of interest
    #gdf_mask = gdf_mask[gdf_mask.intersects(gdf_aoi.unary_union)]
    #gdf_mask_ed = gdf_mask_ed[gdf_mask_ed.intersects(gdf_aoi.unary_union)]

    # Clip mask to area of interest
    #gdf_mask = gpd.overlay(gdf_mask, gdf_aoi, how='intersection')
    #gdf_mask_ed = gpd.overlay(gdf_mask_ed, gdf_aoi, how='intersection')

    # Get tiles that intersect with mask
    #gdf_tiles = gdf_tiles[gdf_tiles.intersects(gdf_mask_ed.unary_union)]

    # Select tiles within bounds (France)
    #bounds = [-5.0, 45.5, 0, 47.5]
    #gdf_tiles = gdf_tiles.cx[bounds[0]:bounds[2], bounds[1]:bounds[3]]

    # Select specific tile
    #gdf_tiles = gdf_tiles[gdf_tiles['name'] == 'z10_x507_y363']
    #gdf_tiles = gdf_tiles[gdf_tiles['name'] == 'z10_x528_y331']

    # filter the GDF on specific criteria related to the intertidal coverage & distance to a GTSM station
    gdf_tiles_red = gdf_tiles[gdf_tiles["intertidal_coverage_ed"]*100 > 0] # percentages
    gdf_tiles_red = gdf_tiles_red[gdf_tiles_red["intertidal_coverage"]*100 >= 1] # percentages
    gdf_tiles_red = gdf_tiles_red[gdf_tiles_red["nearest_station_distance"] <= 37000] # m, 37000 is at Z10 at most on the corner-point of the adjacent tile from the centroid
    
    # count number of occurences ids in reg_regions
    #print(gdf_tiles_red['ref_region'].value_counts())

    # select specific area
    gdf_tiles_red = gdf_tiles_red[gdf_tiles_red["ref_region"] == project_name]

    # Sort tiles based on intertidal coverage
    gdf_tiles_red = gdf_tiles_red.sort_values(by='intertidal_coverage_ed', ascending=False)

    # Print tiles
    print('Number of tiles: {}'.format(len(gdf_tiles_red)))
    gdf_tiles_red.head(5)

    # put to gdf_tiles
    gdf_tiles = gdf_tiles_red

if run_mode == 'global' and project_name == "failed":
    print("Opening a seperate df created in 96_global_coverage, containing the failed tiles for the default run")
    gdf_tiles = gpd.read_parquet(os.path.join(file_path_progress, "failed_tiles_default_run_no_RAR_and_GIC.parquet"))

Number of tiles: 48


In [20]:
#gdf_tiles = gdf_tiles[860:]

In [21]:
# for idx, i in enumerate(gdf_tiles.name):
#     print(idx, i)
gdf_tiles

,name,id,tx,ty,zoom,geometry,path,ref_region,area_perc_org,area_perc_red,nearest_station_id,nearest_station_longitude,nearest_station_latitude,nearest_station_distance,intertidal_coverage,intertidal_coverage_ed
21205,z10_x527_y332,658,527.0,332.0,10,"POLYGON ((5.27344 53.12041, 5.62500 53.12041, ...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,NEU,57.48,57.27,station 21703,5.310059,53.195801,9842.931261,0.575023,0.573314
19827,z10_x528_y331,699,528.0,331.0,10,"POLYGON ((5.62500 53.33087, 5.97656 53.33087, ...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,NEU,40.07,39.16,station 19366,5.749512,53.415527,4064.036790,0.405635,0.391884
20394,z10_x527_y331,657,527.0,331.0,10,"POLYGON ((5.27344 53.33087, 5.62500 53.33087, ...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,NEU,39.49,37.99,station 03947,5.464587,53.438670,1074.042036,0.396725,0.381588
30691,z10_x465_y452,275,465.0,452.0,10,"POLYGON ((-16.52344 20.30342, -16.17187 20.303...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,SAH,18.91,18.34,station 12510,-16.299620,20.345940,14476.799178,0.201735,0.183972
22155,z10_x362_y497,2372,362.0,497.0,10,"POLYGON ((-52.73437 4.91583, -52.38281 4.91583...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,NSA,15.50,15.12,station 19035,-52.631836,5.258789,20352.216226,0.157001,0.151244
30668,z10_x463_y453,178,463.0,453.0,10,"POLYGON ((-17.22656 19.97335, -16.87500 19.973...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,SAH,13.05,12.57,station 12511,-16.601500,20.617500,70932.433386,0.130457,0.126361
30679,z10_x464_y454,228,464.0,454.0,10,"POLYGON ((-16.87500 19.64259, -16.52344 19.642...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,SAH,11.00,10.59,station 12508,-16.447080,19.618750,33753.449700,0.110011,0.106138
30676,z10_x464_y451,225,464.0,451.0,10,"POLYGON ((-16.87500 20.63278, -16.52344 20.632...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,SAH,11.04,10.41,station 12511,-16.601500,20.617500,22409.432337,0.113255,0.104133
30680,z10_x464_y455,229,464.0,455.0,10,"POLYGON ((-16.87500 19.31114, -16.52344 19.311...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,SAH,10.17,9.77,station 17758,-16.489060,19.418920,22957.982907,0.101718,0.098421
30669,z10_x463_y454,179,463.0,454.0,10,"POLYGON ((-17.22656 19.64259, -16.87500 19.642...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,SAH,9.98,9.71,station 12508,-16.447080,19.618750,66604.721117,0.099846,0.097140


In [22]:
# Plot area of interest
# m = folium.Map(location=[gdf_aoi.centroid.y, gdf_aoi.centroid.x], zoom_start=6)
# m = gdf_aoi.explore(m=m, style_kwds={'color': 'red', 'fillOpacity': 0.2}, name='Area of Interest', tooltip=False)
# m = gdf_mask.explore(m=m, style_kwds={'color': 'blue', 'fillOpacity': 0.2}, name='Mask', tooltip=False)
# m = gdf_mask_ed.explore(m=m, style_kwds={'color': 'purple', 'fillOpacity': 0.2}, name='Mask Erosion Dilation', tooltip=False)
# m = gdf_tiles.explore(m=m, cmap='Greens', column='intertidal_coverage_ed', name='Tiles', vmin=0, vmax=np.percentile(gdf_tiles['intertidal_coverage_ed'], 98), tooltip=['id', 'name', 'intertidal_coverage_ed'], 
#                          legend=True)
# folium.LayerControl().add_to(m)
#m

### 5. Compute Satellite Derived Bathymetry

In [23]:
# functions to compute sub & intertidal bathymetry proxies based on standardized SlippyMap tiling practice
# functions taken from: https://github.com/openearth/eo-bathymetry/blob/master/notebooks/rws-bathymetry/export_bathymetry.ipynb
# resembles similar behaviour as in https://github.com/openearth/eo-bathymetry-functions but slightly adjusted for local study 

# Packages
from typing import Optional, List, Dict, Any
from logging import Logger, getLogger
from googleapiclient.discovery import build
from re import sub
from ctypes import ArgumentError
from functools import partial
from dateutil.parser import parse

logger: Logger = getLogger(__name__)

def get_tile_intertidal_bathymetry(tile: ee.Feature, start: ee.String, stop: ee.String) -> ee.Image:
    """
    Get intertidal bathymetry based on tile geometry.
    Server-side compliant for GEE.

    args:
        tile (ee.Feature): tile geometry used to obtain bathymetry.
        start (ee.String): start date in YYYY-MM-dd format.
        stop (ee.String): stop date in YYYY-MM-dd format.
    
    returns:
        ee.Image: image containing intertidal bathymetry covering tile.
    """

    bounds: ee.Geometry = ee.Feature(tile).geometry().bounds(1)
    sdb: Bathymetry = Bathymetry()
    zoom: ee.String = ee.String(tile.get("zoom"))
    tx: ee.String = ee.String(tile.get("tx"))
    ty: ee.String = ee.String(tile.get("ty"))
    tile_name: ee.String = ee.String("z").cat(zoom).cat("_x").cat(tx).cat("_y").cat(ty).replace("\.\d+", "", "g")
    img_fullname: ee.String = ee.String(tile_name).cat("_t").cat(ee.Date(start).millis().format())
        
    image: ee.Image = sdb.compute_intertidal_depth(
        bounds=bounds,
        start=start,
        stop=stop,
        scale=tiler.zoom_to_scale(ee.Number.parse(tile.get("zoom"))).multiply(5), # scale to search for clean images
        # missions=['S2', 'L8'],
        # filter: ee.Filter.dayOfYear(7*30, 9*30), # summer-only
        filter_masked=False, 
        tile=tile,
        # filterMaskedFraction = 0.5,
        # skip_scene_boundary_fix=False,
        # skip_neighborhood_search=False,
        neighborhood_search_parameters={"erosion": 0, "dilation": 0, "weight": 50},
        bounds_buffer=0,
        water_index_min=-0.05,
        water_index_max=0.15,
        # lowerCdfBoundary=45,
        # upperCdfBoundary=50,
        #cloud_frequency_threshold_data=0.3, #ADJUSTED FOR FAILED IMAGES, default is 0.15!
        clip = True,
        mosaic_by_day = True
    )# .reproject(ee.Projection("EPSG:3857").atScale(90))

    image = image.set(
        "fullname", img_fullname,
        "system:time_start", ee.Date(start).millis(),
        "system:time_stop", ee.Date(stop).millis(),
        "zoom", zoom,
        "tx", tx,
        "ty", ty
    )

    return image

def tile_to_asset(
    image: ee.Image,
    tile: ee.Feature,
    export_scale: int,
    asset_path_prefix: str,
    asset_name: str,
    overwrite: bool
) -> Optional[ee.batch.Task]:
    
    asset_id: str = f"{asset_path_prefix}/{asset_name}"
    asset: Dict[str, Any] = ee.data.getInfo(asset_id)
    if overwrite and asset:
        logger.info(f"deleting asset {asset}")
        ee.data.deleteAsset(asset_id)
    elif asset:
        logger.info(f"asset {asset} already exists, skipping {asset_name}")
        return
    task: ee.batch.Task = ee.batch.Export.image.toAsset(
        image,
        assetId=asset_id,
        description=asset_name,
        region=tile.geometry(),
        scale=export_scale,
        maxPixels= 1e10
    )
    task.start()
    logger.info(f"exporting {asset_name} to {asset_id}")

def tile_to_cloud_storage(
    image: ee.Image,
    tile: ee.Feature,
    crs: str,
    export_scale: int,
    bucket: str,
    bucket_path: str,
    overwrite: bool
) -> Optional[ee.batch.Task]:
    with build('storage', 'v1') as storage:
        res = storage.objects().list(bucket=bucket, prefix="/".join(bucket_path.split("/")[:-1])).execute()
    if not overwrite:
        try:
            object_exists = any(map(lambda item: item.get("name").startswith(bucket_path), res.get("items")))
        except AttributeError:
            object_exists = False
        if object_exists:
            logger.info(f"object {bucket_path} already exists in bucket {bucket}, skipping")
            return
        
    task: ee.batch.Task = ee.batch.Export.image.toCloudStorage(
        image,
        bucket=bucket,
        description=bucket_path.replace("/", "_"),
        fileNamePrefix=bucket_path,
        region=tile.geometry(),
        scale=export_scale,
        crs=crs,
        fileFormat='GeoTIFF',
        formatOptions= {'cloudOptimized': True}, # enables easy QGIS plotting
        maxPixels= 1e10
    )
    task.start()
    return task

def metadata_to_cloud_storage(
    image: ee.Image,
    tile: ee.Feature,
    bucket: str,
    bucket_path: str,
    overwrite: bool
) -> Optional[ee.batch.Task]:
    with build('storage', 'v1') as storage:
        res = storage.objects().list(bucket=bucket, prefix="/".join(bucket_path.split("/")[:-1])).execute()
    
    if not overwrite:
        try:
            object_exists = any(map(lambda item: item.get("name").startswith(bucket_path), res.get("items")))
        except AttributeError:
            object_exists = False
        if object_exists:
            logger.info(f"object {bucket_path} already exists in bucket {bucket}, skipping")
            return
    
    meta_feature = ee.Feature(None, image.toDictionary().set("tx", tile.get("tx")).set("ty", tile.get("ty")))

    task: ee.batch.Task = ee.batch.Export.table.toCloudStorage(
        ee.FeatureCollection(meta_feature),
        bucket=bucket,
        description=bucket_path.replace("/", "_"),
        fileNamePrefix=bucket_path,
        fileFormat='csv',
        maxVertices=0
    )
    task.start()
    return task

def export_sdb_tiles(
    sink: str,
    tile_list: ee.List,
    num_tiles: int,
    export_scale: int,
    crs: str,
    sdb_tiles: ee.ImageCollection,
    name_suffix: str,
    mode: str,
    task_list: List[ee.batch.Task],
    overwrite: bool,
    bucket: Optional[str] = None
) -> List[ee.batch.Task]:
    """
    Export list of tiled images containing sub or intertidal tidal bathymetry. Fires off the tasks and adds to the list of tasks.
    based on: https://github.com/gee-community/gee_tools/blob/master/geetools/batch/imagecollection.py#L166

    args:
        sink (str): type of data sink to export to. Viable options are: "asset" and "cloud".
        tile_list (ee.List): list of tile features.
        num_tiles (int): number of tiles in `tile_list`.
        scale (int): scale of the export product.
        sdb_tiles (ee.ImageCollection): collection of subtidal bathymetry images corresponding
            to input tiles.
        name_suffix (str): unique identifier after tile statistics.
        task_list (List[ee.batch.Task]): list of tasks, adds tasks created to this list.
        overwrite (bool): whether to overwrite the current assets under the same `asset_path`.
        bucket (str): Bucket where the data is stored. Only used when sink = "cloud"
    
    returns:
        List[ee.batch.Task]: list of started tasks

    """
    if sink == "asset":
        user_name: str = ee.data.getAssetRoots()[0]["id"].split("/")[-1]
        asset_path_prefix: str = f"users/{user_name}/eo-bathymetry"
        ee.data.create_assets(asset_ids=[asset_path_prefix], asset_type="Folder", mk_parents=True)
    
    for i in range(num_tiles):
        # get tile
        temp_tile: ee.Feature = ee.Feature(tile_list.get(i))
        tile_metadata: Dict[str, Any] = temp_tile.getInfo()["properties"]
        tx: str = tile_metadata["tx"]
        ty: str = tile_metadata["ty"]
        zoom: str = tile_metadata["zoom"]
        # filter imagecollection based on tile
        filtered_ic: ee.ImageCollection = sdb_tiles \
            .filterMetadata("tx", "equals", tx) \
            .filterMetadata("ty", "equals", ty) \
            .filterMetadata("zoom", "equals", zoom)
        # if filtered correctly, only a single image remains
        img: ee.Image = ee.Image(filtered_ic.first())  # have to cast here
        img_name: str = sub(r"\.\d+", "", f"{mode}/z{zoom}/x{tx}/y{ty}/") + name_suffix 
        print("Submitting task for tile: ", img_name)
        # Export images
        if sink == "asset":  # Replace with case / switch in python 3.10
            task_img: Optional[ee.batch.Task] = tile_to_asset(
                image=img,
                tile=temp_tile,
                export_scale=export_scale,
                asset_path_prefix=asset_path_prefix,
                asset_name=img_name.replace("/","_"),
                overwrite=overwrite
            )
            if task_img: task_list.append(task_img)
        elif sink == "cloud":
            if not bucket:
                raise ArgumentError("Sink option requires \"bucket\" arg.")
            task_img: ee.batch.Task = tile_to_cloud_storage(
                image=img,
                tile=temp_tile,
                export_scale=export_scale,
                crs=crs, 
                bucket=bucket,
                bucket_path=img_name,
                overwrite=overwrite
            )

            task_meta: ee.batch.Task = metadata_to_cloud_storage(
                image=img,
                tile=temp_tile,
                bucket=bucket,
                bucket_path=sub(r"\.\d+", "", f"{mode}_meta/z{zoom}/x{tx}/y{ty}/") + name_suffix,
                overwrite=overwrite
            )
        else:
            raise ArgumentError("unrecognized data sink: {sink}")
        task_list.append(task_img)
        task_list.append(task_meta)
    return task_list

def export_tiles(
    sink: str,
    mode: str,
    geometry: ee.Geometry,
    zoom: int,
    start: str,
    stop: str,
    scale: Optional[float] = None,
    crs: str = "EPSG:4326",
    buf_pix: int = 0,
    step_months: int = 3,
    window_months: int = 24,
    overwrite: bool = False,
    bucket: Optional[str] = None
) -> None:
    """
    From a geometry, creates tiles of input zoom level, calculates subtidal bathymetry in those
    tiles, and exports those tiles.

    args:
        sink (str): type of data sink to export to. Viable options are: "asset" and "cloud".
        mode (str): either "subtidal" or "intertidal" for select type of bathymetry to export.
        geometry (ee.Geometry): geometry of the area of interest.
        zoom (int): zoom level of the to-be-exported tiles.
        start (ee.String): start date in YYYY-MM-dd format.
        stop (ee.String): stop date in YYYY-MM-dd format.
        scale Optional(float): scale of the product to be exported. Defaults tiler.zoom_to_scale(zoom).getInfo().
        crs (str): projection of the output image.
        buf_pix (int): buffer around the tile (in pixels).
        step_months (int): steps with which to roll the window over which the subtidal bathymetry
            is calculated.
        windows_months (int): number of months over which the bathymetry is calculated.
    """

    # Function to create a window
    def generate_monthly_windows(start, stop, window_months, step_months):
        start_date = parse(start)
        stop_date = parse(stop)

        # Convert to ee.Date
        ee_start = ee.Date(start)
        ee_stop = ee.Date(stop)

        # Generate list of window start dates
        def generate_windows(n):
            start_dt = ee_start.advance(n.multiply(step_months), 'month')
            end_dt = start_dt.advance(window_months, 'month')
            return ee.Algorithms.If(
                end_dt.millis().lte(ee_stop.millis()),
                ee.Dictionary({
                    'start': start_dt.format('YYYY-MM-dd'),
                    'stop': end_dt.format('YYYY-MM-dd')
                }),
                None  # Skip if window end goes past stop date
            )

        # Compute max number of windows
        total_months = (stop_date.year - start_date.year) * 12 + (stop_date.month - start_date.month)
        max_windows = int((total_months - window_months) / step_months) + 1

        dates = ee.List.sequence(0, max_windows - 1) \
            .map(lambda n: generate_windows(ee.Number(n))) \
            .removeAll([None])  # Remove None entries

        return ee.List(dates)  # Convert to native Python list of dicts
    
    dates = generate_monthly_windows(start, stop, window_months, step_months)
    #dates = ee.List([dates.get(0)]) #ADJUSTED TO SELECT FIRST DATE ONLY
    
    # Get tiles
    tile: ee.Feature =  ee.Feature(geometry.buffer(buf_pix*scale/111120, ee.ErrorMargin((buf_pix*scale*0.01)/111120, 'projected'), proj="EPSG:4326"))
    tiles: ee.FeatureCollection = ee.FeatureCollection(tile) #ADJUSTED TO SELECT SINGLE TILE

    # Get number of tiles
    num_tiles: int = tiles.size().getInfo() # tile_list #ADDED TO UPDATE EXPORT FOR ONLY CALIBRATED IMAGES
    if num_tiles == 0:
        print("GTSM collection empty!")
        return

    # Get scale (if not specified)
    if scale == None:
        scale: float = tiler.zoom_to_scale(zoom).getInfo() # not specified, defaults to pre-set float
    
    # Get tasks
    task_list: List[ee.batch.Task] = []
    for date in dates.getInfo():
        print(date)
        if "subtidal" in mode:
            print('Subtidal mode not available')
        elif "intertidal" in mode:
            # Get subtidal bathymetry for tiles
            sdb_tiles: ee.ImageCollection = tiles.map(
                lambda tile: get_tile_intertidal_bathymetry(
                    tile=tile,
                    start=ee.String(date["start"]),
                    stop=ee.String(date["stop"])
                )#.clip(geometry)#.select('ndwi').rename('water_score') # clip individual tiles to match geometry of aoi, select ndwi and rename
            )

        # Convert tiles to list
        tile_list: ee.List = tiles.toList(num_tiles)

        #Export tiles
        task_list = export_sdb_tiles(
            sink=sink,
            tile_list=tile_list, # tile_list_up
            num_tiles=num_tiles,
            mode=mode,
            export_scale=scale,
            crs=crs,
            sdb_tiles=sdb_tiles, # sdb_tiles_up
            name_suffix=f"t{date['start']}_{date['stop']}_{scale}m",
            task_list=task_list,
            overwrite=overwrite,
            bucket=bucket
        )

    return task_list 

In [27]:
# Compute intertidal bathymetry for each tile. When tasks are submitted, check progress at:
# https://code.earthengine.google.com/tasks or https://console.cloud.google.com/earth-engine/tasks?project=bathymetry

tasks = []
for idx, row in tqdm(gdf_tiles.iterrows(), total=gdf_tiles.shape[0]):
    # Get tile
    ee_tile = ee.Geometry(row['geometry'].__geo_interface__, gdf_tiles.crs.to_string(), False)

    # Get properties
    ee_properties = {'tx': ee.String(str(row['tx'])), 'ty': ee.String(str(row['ty'])), 'zoom': ee.String(str(row['zoom'])),
                     'nearest_station_id': ee.String(row['nearest_station_id']), 'nearest_station_distance': ee.Number(row['nearest_station_distance']),
                     'nearest_station_latitude': ee.Number(row['nearest_station_latitude']), 'nearest_station_longitude': ee.Number(row['nearest_station_longitude'])}
    
    # Create feature
    ee_feature = ee.Feature(ee_tile).set(ee_properties)

    # Export tiles
    task = export_tiles(sink='cloud', mode=mode, geometry=ee_feature, zoom=zoom_level, start=start_date, stop=stop_date,
                        scale=scale, crs=crs, buf_pix=5, step_months=compo_int, window_months=compo_len, overwrite=True, bucket=bucket)
    
    # Append taks
    tasks.append(task)

# Get start time
start_time = time.time()

# save the task list as a pickle file
with open(os.path.join(file_path_progress, ("tasks_" + project_name +".pkl")), "wb") as f:
    pickle.dump(tasks, f)

  0%|          | 0/48 [00:00<?, ?it/s]

{'start': '2019-01-01', 'stop': '2020-01-01'}
Submitting task for tile:  intertidal_improved_10m_dynamics/z10/x527/y332/t2019-01-01_2020-01-01_10m
{'start': '2020-01-01', 'stop': '2021-01-01'}
Submitting task for tile:  intertidal_improved_10m_dynamics/z10/x527/y332/t2020-01-01_2021-01-01_10m
{'start': '2021-01-01', 'stop': '2022-01-01'}
Submitting task for tile:  intertidal_improved_10m_dynamics/z10/x527/y332/t2021-01-01_2022-01-01_10m
{'start': '2022-01-01', 'stop': '2023-01-01'}
Submitting task for tile:  intertidal_improved_10m_dynamics/z10/x527/y332/t2022-01-01_2023-01-01_10m
{'start': '2023-01-01', 'stop': '2024-01-01'}


C:\Users\kras\Documents\GitHub\ee-packages-py\eepackages\applications\bathymetry.py:615: UserWarning: Warning: No GTSM collection found for year {year}, skipping.
  warnings.warn("Warning: No GTSM collection found for year {year}, skipping.")


Submitting task for tile:  intertidal_improved_10m_dynamics/z10/x527/y332/t2023-01-01_2024-01-01_10m


  2%|▏         | 1/48 [00:25<20:01, 25.56s/it]

...

 98%|█████████▊| 47/48 [18:11<00:26, 26.82s/it]

{'start': '2019-01-01', 'stop': '2020-01-01'}
Submitting task for tile:  intertidal_improved_10m_dynamics/z10/x508/y368/t2019-01-01_2020-01-01_10m
{'start': '2020-01-01', 'stop': '2021-01-01'}
Submitting task for tile:  intertidal_improved_10m_dynamics/z10/x508/y368/t2020-01-01_2021-01-01_10m
{'start': '2021-01-01', 'stop': '2022-01-01'}
Submitting task for tile:  intertidal_improved_10m_dynamics/z10/x508/y368/t2021-01-01_2022-01-01_10m
{'start': '2022-01-01', 'stop': '2023-01-01'}
Submitting task for tile:  intertidal_improved_10m_dynamics/z10/x508/y368/t2022-01-01_2023-01-01_10m
{'start': '2023-01-01', 'stop': '2024-01-01'}


C:\Users\kras\Documents\GitHub\ee-packages-py\eepackages\applications\bathymetry.py:615: UserWarning: Warning: No GTSM collection found for year {year}, skipping.
  warnings.warn("Warning: No GTSM collection found for year {year}, skipping.")


Submitting task for tile:  intertidal_improved_10m_dynamics/z10/x508/y368/t2023-01-01_2024-01-01_10m


100%|██████████| 48/48 [18:37<00:00, 23.28s/it]


In [26]:
project_name = "dynamics_10m"

In [302]:
# save the task list as a pickle file
# with open(os.path.join(file_path_progress, ("tasks_" + project_name +"1.pkl")), "wb") as f:
#     pickle.dump(tasks, f)

In [31]:
# Monitor tasks
project_name = "dynamics_10m"

# open the task list as a pickle file
with open(os.path.join(file_path_progress, ("tasks_" + project_name +".pkl")), "rb") as f:
    tasks = pickle.load(f)

n_tasks_failed, n_tasks_complete, n_tasks = 0, 0, 1
while n_tasks_failed + n_tasks_complete < n_tasks:
    # Get number of tasks
    n_tasks = len([task for tasks_ in tasks for task in tasks_])
    
    # Get task statuses
    task_statuses = [task.status() for tasks_ in tasks for task in tasks_]

    # Get number of tasks running, completed and failed
    n_tasks_ready = sum([task_status['state'] == 'READY' for task_status in task_statuses])
    n_tasks_running = sum([task_status['state'] == 'RUNNING' for task_status in task_statuses])
    n_tasks_complete = sum([task_status['state'] == 'COMPLETED' for task_status in task_statuses])
    n_tasks_failed = sum([task_status['state'] == 'FAILED' for task_status in task_statuses])

    # Get time elapsed
    time_elapsed = time.time() - start_time

    # Print tasks
    print('Tasks: {} ready, {} running, {} complete, {} failed (after {:.2f} minutes)'.format(n_tasks_ready, n_tasks_running, n_tasks_complete, n_tasks_failed, time_elapsed / 60), end='\r')

    # Wait for 10 seconds
    time.sleep(10)

# Print tasks
print('Tasks: ready {}, {} running, {} complete, {} failed (after {:.2f} minutes)'.format(n_tasks_ready, n_tasks_running, n_tasks_complete, n_tasks_failed, time_elapsed / 60))

KeyboardInterrupt: 